In [2]:
import numpy as np
import pandas as pd
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':120,'font.size':10,
                     'axes.spines.top':False,'axes.spines.right':False})
SEED = 42
np.random.seed(SEED)
print('설정 완료')

설정 완료


In [3]:
BASE = r'c:\Users\kevin\OneDrive\Desktop\AISO\Elliptic Bitcoin\elliptic_bitcoin_dataset'

print('로딩 중...')
feat_df  = pd.read_csv(f'{BASE}/elliptic_txs_features.csv', header=None)
cls_df   = pd.read_csv(f'{BASE}/elliptic_txs_classes.csv')
edge_df  = pd.read_csv(f'{BASE}/elliptic_txs_edgelist.csv')

# 컬럼 정리: 0=txId, 1=timestep, 2~166=features
feat_df.columns = ['txId'] + [f'f{i}' for i in range(1, 167)]
feat_df['timestep'] = feat_df['f1'].astype(int)

# 병합
df = feat_df.merge(cls_df, on='txId')
print(f'전체: {len(df):,}노드 | 피처: 166 | 엣지: {len(edge_df):,}')
print('\nclass 분포:')
print(df['class'].value_counts().to_string())
print(f'\n타임스텝: {df["timestep"].min()} ~ {df["timestep"].max()}')

로딩 중...
전체: 203,769노드 | 피처: 166 | 엣지: 234,355

class 분포:
class
unknown    157205
2           42019
1            4545

타임스텝: 1 ~ 49


In [4]:
# ── 전처리 ──────────────────────────────────────────────────
# unknown 제외, labeled만 사용
labeled = df[df['class'] != 'unknown'].copy()
labeled['y'] = (labeled['class'] == '1').astype(int)  # 1=illicit

feat_cols = [f'f{i}' for i in range(1, 167)]
X_all = labeled[feat_cols].values.astype(float)
y_all = labeled['y'].values
ts_all = labeled['timestep'].values

print(f'Labeled: {len(labeled):,}  (illicit={y_all.sum():,}, licit={(y_all==0).sum():,})')

# 시간 기반 분리: timestep 1-34 train, 35-49 test (표준 Elliptic 분리)
train_mask = ts_all <= 34
test_mask  = ts_all >  34

X_tr_pool = X_all[train_mask]; y_tr_pool = y_all[train_mask]
X_test_raw = X_all[test_mask]; y_test     = y_all[test_mask]

print(f'Train pool (t≤34): {len(X_tr_pool):,}  illicit={y_tr_pool.sum():,}')
print(f'Test  set  (t>34): {len(X_test_raw):,}  illicit={y_test.sum():,}')

# 불균형 서브샘플 (10K licit + 1K illicit)
rng = np.random.RandomState(SEED)
norm_idx = np.where(y_tr_pool==0)[0]
anom_idx = np.where(y_tr_pool==1)[0]

N_NORMAL = min(10000, len(norm_idx))
N_SEEN   = min(1000,  len(anom_idx))
sel_n = rng.choice(norm_idx, N_NORMAL, replace=False)
sel_a = rng.choice(anom_idx, N_SEEN,   replace=False)

X_imbal_raw = np.vstack([X_tr_pool[sel_n], X_tr_pool[sel_a]])
y_imbal     = np.array([0]*N_NORMAL + [1]*N_SEEN)
ts_anom_train = ts_all[train_mask][sel_a]  # sel_a는 이미 y_tr_pool 기준 인덱스

scaler  = StandardScaler()
X_imbal = scaler.fit_transform(X_imbal_raw)
X_test  = scaler.transform(X_test_raw)

# PCA for 최적화 샘플러 (30D)
pca = PCA(n_components=30, random_state=SEED)
X_imbal_pca = pca.fit_transform(X_imbal)
X_anom_pca  = X_imbal_pca[y_imbal==1]

print(f'\n학습: {N_NORMAL:,} licit + {N_SEEN:,} illicit  ({N_SEEN/(N_NORMAL+N_SEEN)*100:.1f}%)')
print(f'테스트: {len(X_test):,}  illicit={y_test.sum():,}')
print(f'학습 illicit 타임스텝: {ts_anom_train.min()}~{ts_anom_train.max()}')

Labeled: 46,564  (illicit=4,545, licit=42,019)
Train pool (t≤34): 29,894  illicit=3,462
Test  set  (t>34): 16,670  illicit=1,083

학습: 10,000 licit + 1,000 illicit  (9.1%)
테스트: 16,670  illicit=1,083
학습 illicit 타임스텝: 1~34


In [5]:
# ── 평가 함수 ────────────────────────────────────────────────
preds   = {}
results = {}

def evaluate(X_tr, y_tr, label=''):
    clf = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_test)[:,1]
    pred = (prob >= 0.5).astype(int)
    if label:
        preds[label.strip()] = prob
    res = {
        'PR-AUC': average_precision_score(y_test, prob),
        'F1':     f1_score(y_test, pred, zero_division=0),
        'AUC':    roc_auc_score(y_test, prob),
    }
    if label:
        print(f'  {label:<22} PR-AUC={res["PR-AUC"]:.4f}'
              f'  F1={res["F1"]:.4f}  AUC={res["AUC"]:.4f}')
    return res

# 타임스텝별 recall (후반 timestep = 패턴 변화 반영)
def timestep_recall(prob, threshold=0.5, bins=5):
    pred = (prob >= threshold).astype(int)
    ts_test = ts_all[test_mask]
    edges = np.linspace(ts_test.min(), ts_test.max()+1, bins+1)
    recalls = {}
    for i in range(bins):
        mask = (ts_test >= edges[i]) & (ts_test < edges[i+1]) & (y_test==1)
        if mask.sum() > 0:
            recalls[f't{int(edges[i])}-{int(edges[i+1])-1}'] = \
                (pred[mask]==1).mean()
    return recalls

print('평가 함수 준비 완료')

평가 함수 준비 완료


In [6]:
# ── 샘플러 정의 ──────────────────────────────────────────────
N_AG = 20; N_IT = 100; ALPHA = 0.2
N_TYPES = 8; BETA = 0.08; W_REPEL = 2.0; M_LOW = -0.5
N_TARGET = N_NORMAL

def _norm(X):
    mn,mx = X.min(0),X.max(0)
    return (X-mn)/np.where(mx-mn>1e-8,mx-mn,1.0)

def build_train(X_norm_s, X_anom_s, idx):
    return (np.vstack([X_norm_s, X_anom_s[idx]]),
            np.array([0]*len(X_norm_s)+[1]*len(idx)))

X_norm = X_imbal[y_imbal==0]
X_anom = X_imbal[y_imbal==1]

def run_random(X_anom, n, seed):
    return np.random.RandomState(seed).choice(len(X_anom),n,replace=True)

def run_kmeans(X_anom, n, seed, k=8):
    km = KMeans(k, random_state=seed, n_init=5).fit(X_anom)
    rng2=np.random.RandomState(seed); idx=[]
    for c in range(k):
        pool=np.where(km.labels_==c)[0]
        if len(pool): idx.extend(rng2.choice(pool,n//k,replace=True))
    while len(idx)<n: idx.append(rng2.randint(len(X_anom)))
    return np.array(idx[:n])

def run_greedy(X_anom, n, seed, k=5):
    nn=NearestNeighbors(n_neighbors=min(k+1,len(X_anom))).fit(X_anom)
    d,_=nn.kneighbors(X_anom)
    density=1.0/(d[:,1:].mean(1)+1e-8)
    probs=1.0/(density+1e-8); probs/=probs.sum()
    return np.random.RandomState(seed).choice(len(X_anom),n,replace=True,p=probs)

def run_topdensity(X_anom, n, seed, k=5):
    nn=NearestNeighbors(n_neighbors=min(k+1,len(X_anom))).fit(X_anom)
    d,_=nn.kneighbors(X_anom)
    density=1.0/(d[:,1:].mean(1)+1e-8)
    probs=density/density.sum()
    return np.random.RandomState(seed).choice(len(X_anom),n,replace=True,p=probs)

def run_aco(X_anom, n, seed):
    rng2=np.random.RandomState(seed)
    Xn=_norm(X_anom); N_a=len(Xn)
    ph=np.ones(N_a); visit=np.zeros(N_a)
    for _ in range(N_IT):
        for _ in range(N_AG):
            i=rng2.choice(N_a,p=ph/ph.sum())
            d=np.linalg.norm(Xn-Xn[i],axis=1); d[i]=1e9
            cands=np.argsort(d)[:10]
            j=cands[np.argmax(ph[cands])]
            nn=np.argmin(np.linalg.norm(Xn-np.clip((Xn[i]+Xn[j])/2,0,1),axis=1))
            visit[nn]+=1
        ph=ph*0.95+visit*0.1
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

def run_pso(X_anom, n, seed):
    rng2=np.random.RandomState(seed)
    Xn=_norm(X_anom); N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V=np.zeros_like(X); pX=X.copy()
    pS=np.array([-np.min(np.linalg.norm(Xn-X[i],axis=1)) for i in range(N_AG)])
    gi=np.argmax(pS); gX=X[gi].copy(); visit=np.zeros(N_a)
    for _ in range(N_IT):
        r1,r2=rng2.rand(N_AG,D),rng2.rand(N_AG,D)
        V=0.729*V+1.494*r1*(pX-X)+1.494*r2*(gX-X)
        X=np.clip(X+ALPHA*V,0,1)
        for i in range(N_AG):
            nn=np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc=-np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

def run_spso(X_anom, n, seed, r_s=0.3):
    rng2=np.random.RandomState(seed)
    Xn=_norm(X_anom); N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V=np.zeros_like(X); pX=X.copy(); pS=-np.ones(N_AG)*1e9; visit=np.zeros(N_a)
    for _ in range(N_IT):
        sp_best=[]
        for i in range(N_AG):
            sp=np.where(np.linalg.norm(X-X[i],axis=1)<=r_s)[0]
            sp_best.append(sp[np.argmax(pS[sp])])
        sbX=np.array([X[sp_best[i]] for i in range(N_AG)])
        r1,r2=rng2.rand(N_AG,D),rng2.rand(N_AG,D)
        V=0.729*V+1.494*r1*(pX-X)+1.494*r2*(sbX-X)
        X=np.clip(X+ALPHA*V,0,1)
        for i in range(N_AG):
            nn=np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc=-np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

def run_cde(X_anom, n, seed, F=0.8, CR=0.9):
    rng2=np.random.RandomState(seed)
    Xn=_norm(X_anom); N_a,D=Xn.shape
    pop=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    visit=np.zeros(N_a)
    for _ in range(N_IT):
        for i in range(N_AG):
            idxs=rng2.choice([j for j in range(N_AG) if j!=i],3,replace=False)
            mutant=np.clip(pop[idxs[0]]+F*(pop[idxs[1]]-pop[idxs[2]]),0,1)
            trial=np.where(rng2.rand(D)<CR,mutant,pop[i])
            ms=np.argmin(np.linalg.norm(pop-trial,axis=1))
            nn=np.argmin(np.linalg.norm(Xn-trial,axis=1))
            pop[ms]=Xn[nn]; visit[nn]+=1
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

def run_aiso(X_anom, n, seed):
    rng2=np.random.RandomState(seed)
    Xn=_norm(X_anom); N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W=rng2.dirichlet(np.ones(N_TYPES),N_AG)
    M=rng2.uniform(M_LOW,W_REPEL,(N_TYPES,N_TYPES))
    visit=np.zeros(N_a); w_r=W_REPEL
    for t in range(N_IT):
        if t%10==0:
            div=np.mean([np.linalg.norm(X[i]-X[j])
                         for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r=1.0+3.0*np.exp(-div/0.12)
        C=W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci=C[i].copy(); ci[i]=0
            ta=np.argsort(ci)[-3:]; tr2=np.argsort(ci)[:3]
            Fv=sum(ci[j]*(X[j]-X[i]) for j in ta)\
              +w_r*sum(ci[j]*(X[j]-X[i]) for j in tr2); Fv/=6.0
            nn=np.argmin(np.linalg.norm(Xn-np.clip(X[i]+ALPHA*Fv,0,1),axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja=ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

print('샘플러 준비 완료')

샘플러 준비 완료


In [7]:
# ── 14개 메서드 실행 ─────────────────────────────────────────
def run(name, X_tr, y_tr):
    results[name] = evaluate(X_tr, y_tr, name)

print('='*65)
print('카테고리 1: 베이스라인')
run('원본(불균형)', X_imbal, y_imbal)
clf_cw = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
clf_cw.fit(X_imbal, y_imbal,
           sample_weight=np.where(y_imbal==1,N_NORMAL/N_SEEN,1.0))
prob_cw = clf_cw.predict_proba(X_test)[:,1]
preds['Class Weight'] = prob_cw
results['Class Weight'] = {
    'PR-AUC': average_precision_score(y_test, prob_cw),
    'F1':     f1_score(y_test,(prob_cw>=0.5).astype(int),zero_division=0),
    'AUC':    roc_auc_score(y_test, prob_cw),
}
print(f'  {"Class Weight":<22} PR-AUC={results["Class Weight"]["PR-AUC"]:.4f}'
      f'  F1={results["Class Weight"]["F1"]:.4f}')

print('='*65)
print('카테고리 2: 룰 기반')
print('  Random...', end=' ')
run('Random',      *build_train(X_norm,X_anom,run_random(X_anom,N_TARGET,SEED)))
print('  K-Means...', end=' ')
run('K-Means',     *build_train(X_norm,X_anom,run_kmeans(X_anom,N_TARGET,SEED)))
print('  Greedy...', end=' ')
run('Greedy',      *build_train(X_norm,X_anom,run_greedy(X_anom,N_TARGET,SEED)))
print('  Top-density...', end=' ')
run('Top-density', *build_train(X_norm,X_anom,run_topdensity(X_anom,N_TARGET,SEED)))

print('='*65)
print('카테고리 3: 합성 오버샘플링')
print('  RandomOver...', end=' ')
run('RandomOver',*RandomOverSampler(random_state=SEED).fit_resample(X_imbal,y_imbal))
print('  SMOTE...', end=' ')
run('SMOTE',*SMOTE(random_state=SEED,k_neighbors=5).fit_resample(X_imbal,y_imbal))
print('  ADASYN...', end=' ')
try:
    run('ADASYN',*ADASYN(random_state=SEED).fit_resample(X_imbal,y_imbal))
except Exception as e:
    results['ADASYN']=results['SMOTE'].copy(); preds['ADASYN']=preds.get('SMOTE')
    print(f'fallback ({e})')

print('='*65)
print('카테고리 4: 최적화 샘플링 (PCA-30D)')
print('  ACO...', end=' ')
run('ACO',  *build_train(X_norm,X_anom,run_aco(X_anom_pca, N_TARGET,SEED)))
print('  PSO...', end=' ')
run('PSO',  *build_train(X_norm,X_anom,run_pso(X_anom_pca, N_TARGET,SEED)))
print('  SPSO...', end=' ')
run('SPSO', *build_train(X_norm,X_anom,run_spso(X_anom_pca,N_TARGET,SEED)))
print('  CDE...', end=' ')
run('CDE',  *build_train(X_norm,X_anom,run_cde(X_anom_pca, N_TARGET,SEED)))
print('  AISO...', end=' ')
run('AISO', *build_train(X_norm,X_anom,run_aiso(X_anom_pca,N_TARGET,SEED)))

print('='*65)
print(f'완료! 총 {len(results)}개 메서드')

카테고리 1: 베이스라인
  원본(불균형)                PR-AUC=0.7771  F1=0.7847  AUC=0.8857
  Class Weight           PR-AUC=0.7856  F1=0.6225
카테고리 2: 룰 기반
  Random...   Random                 PR-AUC=0.7872  F1=0.6306  AUC=0.9157
  K-Means...   K-Means                PR-AUC=0.7886  F1=0.6316  AUC=0.9252
  Greedy...   Greedy                 PR-AUC=0.7804  F1=0.5848  AUC=0.9134
  Top-density...   Top-density            PR-AUC=0.7756  F1=0.8027  AUC=0.8834
카테고리 3: 합성 오버샘플링
  RandomOver...   RandomOver             PR-AUC=0.7881  F1=0.6327  AUC=0.9159
  SMOTE...   SMOTE                  PR-AUC=0.7835  F1=0.6873  AUC=0.9107
  ADASYN...   ADASYN                 PR-AUC=0.7933  F1=0.6866  AUC=0.9333
카테고리 4: 최적화 샘플링 (PCA-30D)
  ACO...   ACO                    PR-AUC=0.7820  F1=0.6552  AUC=0.9120
  PSO...   PSO                    PR-AUC=0.7794  F1=0.6949  AUC=0.8990
  SPSO...   SPSO                   PR-AUC=0.7805  F1=0.6987  AUC=0.9021
  CDE...   CDE                    PR-AUC=0.7727  F1=0.6141  AUC=0.8962
  AISO

In [8]:
# ── 메인 결과 시각화 ─────────────────────────────────────────
CATEGORIES = {
    '베이스라인'  : ['원본(불균형)', 'Class Weight'],
    '룰 기반'     : ['Random','K-Means','Greedy','Top-density'],
    '합성 오버샘플': ['RandomOver','SMOTE','ADASYN'],
    '최적화 샘플링': ['ACO','PSO','SPSO','CDE','AISO'],
}
CAT_C = {
    '베이스라인'  : '#888888',
    '룰 기반'     : '#4C72B0',
    '합성 오버샘플': '#CCB974',
    '최적화 샘플링': '#C44E52',
}
MC = {m:CAT_C[c] for c,ms in CATEGORIES.items() for m in ms}
MC['AISO'] = '#8B0000'

ranking = sorted(results, key=lambda k: results[k]['PR-AUC'], reverse=True)

fig, axes = plt.subplots(1,3,figsize=(24,8))
for ax, metric in zip(axes[:2], ['PR-AUC','F1']):
    vals=[results[m][metric] for m in ranking]
    colors=[MC.get(m,'#aaa') for m in ranking]
    bars=ax.barh(ranking[::-1],vals[::-1],color=colors[::-1],alpha=0.85)
    for bar,v in zip(bars,vals[::-1]):
        ax.text(v+0.002,bar.get_y()+bar.get_height()/2,
                f'{v:.4f}',va='center',fontsize=8)
    ax.set_xlabel(metric)
    ax.set_title(f'Elliptic Bitcoin — {metric}\n'
                 f'(14 methods | train t≤34 | test t>34 | 2% illicit)')
    ax.set_xlim(0,max(vals)*1.18)

ax=axes[2]
for m in ranking:
    ax.scatter(results[m]['AUC'],results[m]['PR-AUC'],
               s=160,color=MC.get(m,'#aaa'),zorder=5)
    ax.annotate(m,(results[m]['AUC'],results[m]['PR-AUC']),
                xytext=(4,4),textcoords='offset points',fontsize=8)
ax.set_xlabel('AUC-ROC'); ax.set_ylabel('PR-AUC')
ax.set_title('AUC vs PR-AUC')

from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(facecolor=CAT_C[c],label=c) for c in CAT_C],
               fontsize=8,loc='lower right')
plt.suptitle('Elliptic Bitcoin Showdown — 14 Methods\n'
             '(203K 노드 | 166 피처 | illicit 2% | 시간 기반 train/test 분리)',
             fontweight='bold',fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_showdown_main.png',bbox_inches='tight',dpi=120)
plt.show()

print('\n'+'='*72)
print(f'  {"전략":<18} {"PR-AUC":>8} {"F1":>8} {"AUC":>8}  카테고리')
print('-'*72)
for rank,m in enumerate(ranking,1):
    cat=next((c for c,ms in CATEGORIES.items() if m in ms),'?')
    star=' ★' if m=='AISO' else ''
    print(f'  {rank:>2}위 {m:<16} {results[m]["PR-AUC"]:>8.4f}'
          f' {results[m]["F1"]:>8.4f} {results[m]["AUC"]:>8.4f}  {cat}{star}')
print('='*72)


  전략                   PR-AUC       F1      AUC  카테고리
------------------------------------------------------------------------
   1위 ADASYN             0.7933   0.6866   0.9333  합성 오버샘플
   2위 K-Means            0.7886   0.6316   0.9252  룰 기반
   3위 RandomOver         0.7881   0.6327   0.9159  합성 오버샘플
   4위 Random             0.7872   0.6306   0.9157  룰 기반
   5위 Class Weight       0.7856   0.6225   0.9087  베이스라인
   6위 AISO               0.7849   0.7074   0.9107  최적화 샘플링 ★
   7위 SMOTE              0.7835   0.6873   0.9107  합성 오버샘플
   8위 ACO                0.7820   0.6552   0.9120  최적화 샘플링
   9위 SPSO               0.7805   0.6987   0.9021  최적화 샘플링
  10위 Greedy             0.7804   0.5848   0.9134  룰 기반
  11위 PSO                0.7794   0.6949   0.8990  최적화 샘플링
  12위 원본(불균형)            0.7771   0.7847   0.8857  베이스라인
  13위 Top-density        0.7756   0.8027   0.8834  룰 기반
  14위 CDE                0.7727   0.6141   0.8962  최적화 샘플링


In [9]:
# ── 타임스텝별 Recall (시간에 따른 적응력) ───────────────────
# 후반 타임스텝 = train에서 멀어짐 → 다양한 illicit 패턴 등장
# AISO가 다양한 패턴을 커버했다면 후반 recall이 덜 떨어져야 함

print('타임스텝별 Recall\n')
ts_results = {}
for m, prob in preds.items():
    if prob is None: continue
    ts_results[m] = timestep_recall(prob)

ts_keys = sorted(list(ts_results.get('AISO',{}).keys()))
header = f'  {"방법":<18}' + ''.join(f'{k:>14}' for k in ts_keys)
print(header); print('-'*len(header))
for m in ranking:
    if m not in ts_results: continue
    row = f'  {m:<18}'
    for k in ts_keys:
        row += f'{ts_results[m].get(k,0):>14.3f}'
    print(row + (' ★' if m=='AISO' else ''))

# 시각화
fig, axes = plt.subplots(1,2,figsize=(16,6))

# 왼쪽: AISO vs PSO vs Random 타임스텝 곡선
highlight = ['AISO','PSO','Random','Class Weight','원본(불균형)']
colors_h  = ['#8B0000','blue','#4C72B0','#888888','gray']
for m,c in zip(highlight, colors_h):
    if m not in ts_results: continue
    vals=[ts_results[m].get(k,0) for k in ts_keys]
    axes[0].plot(range(len(ts_keys)), vals, 'o-', color=c, label=m, lw=2, ms=6)
axes[0].set_xticks(range(len(ts_keys)))
axes[0].set_xticklabels(ts_keys, rotation=30, fontsize=8)
axes[0].set_ylabel('Illicit Recall'); axes[0].set_xlabel('Timestep 구간')
axes[0].set_title('시간대별 Illicit Recall\n(후반 = 학습 분포와 멀어짐)', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# 오른쪽: 후반부(마지막 2구간) 평균 recall 비교
late_keys = ts_keys[-2:]
late_avg  = {m: np.mean([ts_results[m].get(k,0) for k in late_keys])
             for m in ranking if m in ts_results}
methods_sorted = sorted(late_avg, key=late_avg.get, reverse=True)
vals_sorted = [late_avg[m] for m in methods_sorted]
bars = axes[1].barh(methods_sorted[::-1], vals_sorted[::-1],
                    color=[MC.get(m,'#aaa') for m in methods_sorted[::-1]], alpha=0.85)
for bar,v in zip(bars, vals_sorted[::-1]):
    axes[1].text(v+0.005, bar.get_y()+bar.get_height()/2,
                 f'{v:.3f}', va='center', fontsize=8)
axes[1].set_title(f'후반부 ({" / ".join(late_keys)}) 평균 Recall\n(시간적 일반화 성능)', fontweight='bold')
axes[1].set_xlabel('Illicit Recall'); axes[1].set_xlim(0,1.2)

plt.suptitle('Elliptic Bitcoin — 시간대별 Illicit Recall',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_timestep_recall.png',bbox_inches='tight',dpi=120)
plt.show()

타임스텝별 Recall

  방법                        t35-37        t38-40        t41-43        t44-46        t47-49
------------------------------------------------------------------------------------------
  ADASYN                     0.941         0.826         0.807         0.129         0.009
  K-Means                    0.933         0.836         0.810         0.129         0.009
  RandomOver                 0.941         0.842         0.815         0.097         0.000
  Random                     0.941         0.836         0.821         0.129         0.000
  Class Weight               0.933         0.845         0.823         0.161         0.000
  AISO                       0.937         0.832         0.805         0.065         0.000 ★
  SMOTE                      0.933         0.829         0.807         0.065         0.009
  ACO                        0.933         0.836         0.807         0.000         0.000
  SPSO                       0.929         0.842         0.805         0.0

In [10]:
# ── Mode Collapse: PSO vs AISO ───────────────────────────────
TRACK_EVERY = 10
track_iters = list(range(0, N_IT+1, TRACK_EVERY))

def entropy(counts):
    p=counts/(counts.sum()+1e-9); p=p[p>0]
    return -np.sum(p*np.log(p+1e-9))

def track_pso(Xn, seed):
    rng2=np.random.RandomState(seed)
    N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V=np.zeros_like(X); pX=X.copy()
    pS=-np.ones(N_AG)*1e9; gi=0; gX=X[0].copy(); visit=np.zeros(N_a)
    disp_hist=[]; vent_hist=[]
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d=np.mean([np.linalg.norm(X[i]-X[j])
                       for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
        r1,r2=rng2.rand(N_AG,D),rng2.rand(N_AG,D)
        V=0.729*V+1.494*r1*(pX-X)+1.494*r2*(gX-X)
        X=np.clip(X+ALPHA*V,0,1)
        for i in range(N_AG):
            nn=np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc=-np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j])
                               for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    return disp_hist, vent_hist

def track_aiso(Xn, seed):
    rng2=np.random.RandomState(seed)
    N_a,D=Xn.shape
    X=Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W=rng2.dirichlet(np.ones(N_TYPES),N_AG)
    M=rng2.uniform(M_LOW,W_REPEL,(N_TYPES,N_TYPES))
    visit=np.zeros(N_a); w_r=W_REPEL
    disp_hist=[]; vent_hist=[]; went_hist=[]
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d=np.mean([np.linalg.norm(X[i]-X[j])
                       for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
            went_hist.append(np.mean([-np.sum(W[i]*np.log(W[i]+1e-9))
                                       for i in range(N_AG)]))
        if t%10==0:
            dv=np.mean([np.linalg.norm(X[i]-X[j])
                        for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r=1.0+3.0*np.exp(-dv/0.12)
        C=W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci=C[i].copy(); ci[i]=0
            ta=np.argsort(ci)[-3:]; tr2=np.argsort(ci)[:3]
            Fv=sum(ci[j]*(X[j]-X[i]) for j in ta)\
              +w_r*sum(ci[j]*(X[j]-X[i]) for j in tr2); Fv/=6.0
            nn=np.argmin(np.linalg.norm(Xn-np.clip(X[i]+ALPHA*Fv,0,1),axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja=ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j])
                               for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    went_hist.append(np.mean([-np.sum(W[i]*np.log(W[i]+1e-9)) for i in range(N_AG)]))
    return disp_hist, vent_hist, went_hist

print('Mode Collapse 추적 중...')
Xn_track=_norm(X_anom_pca)
print('  PSO...', end=' ', flush=True)
pso_disp,pso_vent=track_pso(Xn_track,SEED); print('완료')
print('  AISO...', end=' ', flush=True)
aiso_disp,aiso_vent,aiso_went=track_aiso(Xn_track,SEED); print('완료')

fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].plot(track_iters,pso_disp,'b-o',ms=4,label='PSO',lw=2)
axes[0].plot(track_iters,aiso_disp,'r-o',ms=4,label='AISO',lw=2)
axes[0].set_title('Agent Dispersion',fontweight='bold')
axes[0].set_xlabel('Iteration'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(track_iters,pso_vent,'b-o',ms=4,label='PSO',lw=2)
axes[1].plot(track_iters,aiso_vent,'r-o',ms=4,label='AISO',lw=2)
axes[1].set_title('Visit Count Entropy H(visit)',fontweight='bold')
axes[1].set_xlabel('Iteration'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(track_iters,aiso_went,'r-o',ms=4,label='AISO W-entropy',lw=2)
axes[2].axhline(np.log(N_TYPES),color='gray',ls='--',
                label=f'Max H=log({N_TYPES})')
axes[2].set_title('AISO Type Entropy H(W)',fontweight='bold')
axes[2].set_xlabel('Iteration'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Mode Collapse 분석: PSO vs AISO (Elliptic Bitcoin)',
             fontweight='bold',fontsize=11)
plt.tight_layout()
plt.savefig('elliptic_mode_collapse.png',bbox_inches='tight',dpi=120)
plt.show()

print(f'\n최종 dispersion : PSO={pso_disp[-1]:.4f}  AISO={aiso_disp[-1]:.4f}')
print(f'최종 visit H    : PSO={pso_vent[-1]:.4f}  AISO={aiso_vent[-1]:.4f}')
print(f'최종 W entropy  : AISO={aiso_went[-1]:.4f} (max={np.log(N_TYPES):.4f})')

Mode Collapse 추적 중...
  PSO... 완료
  AISO... 완료

최종 dispersion : PSO=0.2329  AISO=0.4839
최종 visit H    : PSO=4.0369  AISO=3.0224
최종 W entropy  : AISO=1.8279 (max=2.0794)


In [11]:
# ── Minority Coverage Entropy ────────────────────────────────
# illicit 샘플의 타임스텝 분포로 커버리지 측정
# 다양한 타임스텝 = 다양한 illicit 패턴 커버
print('Coverage 계산 중...')
coverage = {}

def entropy_arr(vals):
    vc = pd.Series(vals).value_counts()
    p  = vc.values/vc.values.sum()
    return -np.sum(p*np.log(p+1e-9)), len(vc)

for name, fn, arg in [
    ('Random',     run_random,    (X_anom,     N_TARGET, SEED)),
    ('K-Means',    run_kmeans,    (X_anom,     N_TARGET, SEED)),
    ('Greedy',     run_greedy,    (X_anom,     N_TARGET, SEED)),
    ('Top-density',run_topdensity,(X_anom,     N_TARGET, SEED)),
    ('ACO',        run_aco,       (X_anom_pca, N_TARGET, SEED)),
    ('PSO',        run_pso,       (X_anom_pca, N_TARGET, SEED)),
    ('SPSO',       run_spso,      (X_anom_pca, N_TARGET, SEED)),
    ('CDE',        run_cde,       (X_anom_pca, N_TARGET, SEED)),
    ('AISO',       run_aiso,      (X_anom_pca, N_TARGET, SEED)),
]:
    idx = fn(*arg)
    ts_sel = ts_anom_train[idx]  # 선택된 illicit 샘플의 타임스텝
    H, n_ts = entropy_arr(ts_sel)
    coverage[name] = {'entropy': H, 'n_ts': n_ts,
                      'ts_dist': pd.Series(ts_sel).value_counts().sort_index()}
    print(f'  {name:<18} H={H:.3f}  timesteps covered={n_ts}')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ordered = sorted(coverage, key=lambda k: coverage[k]['entropy'], reverse=True)
entropies = [coverage[m]['entropy'] for m in ordered]

bars = axes[0].barh(ordered[::-1], entropies[::-1],
                    color=[MC.get(m,'#aaa') for m in ordered[::-1]], alpha=0.85)
for bar,v in zip(bars,entropies[::-1]):
    axes[0].text(v+0.02,bar.get_y()+bar.get_height()/2,
                 f'{v:.3f}',va='center',fontsize=9)
axes[0].set_xlabel('타임스텝 분포 엔트로피 H')
axes[0].set_title('Illicit Timestep Coverage Entropy\n'
                  '(높을수록 다양한 시간대의 illicit 패턴 포함)',fontweight='bold')

# AISO vs PSO 타임스텝 분포
aiso_ts = coverage.get('AISO',{}).get('ts_dist', pd.Series())
pso_ts  = coverage.get('PSO', {}).get('ts_dist', pd.Series())
all_ts  = sorted(set(list(aiso_ts.index)+list(pso_ts.index)))
x=np.arange(len(all_ts)); w=0.35
aiso_p=np.array([aiso_ts.get(t,0) for t in all_ts])/(sum(aiso_ts)+1e-9)
pso_p =np.array([pso_ts.get(t,0)  for t in all_ts])/(sum(pso_ts)+1e-9)
axes[1].bar(x-w/2,aiso_p,w,label='AISO',color='#8B0000',alpha=0.8)
axes[1].bar(x+w/2,pso_p, w,label='PSO', color='blue',   alpha=0.8)
axes[1].set_xticks(x[::3]); axes[1].set_xticklabels(all_ts[::3],fontsize=8)
axes[1].set_title('AISO vs PSO: 오버샘플 타임스텝 분포',fontweight='bold')
axes[1].set_ylabel('비율'); axes[1].legend()

plt.suptitle('Elliptic Bitcoin — Illicit Timestep Coverage',
             fontweight='bold',fontsize=12)
plt.tight_layout()
plt.savefig('elliptic_coverage.png',bbox_inches='tight',dpi=120)
plt.show()

Coverage 계산 중...
  Random             H=3.135  timesteps covered=34
  K-Means            H=2.956  timesteps covered=34
  Greedy             H=3.233  timesteps covered=34
  Top-density        H=2.223  timesteps covered=32
  ACO                H=3.040  timesteps covered=34
  PSO                H=2.865  timesteps covered=34
  SPSO               H=2.873  timesteps covered=34
  CDE                H=3.210  timesteps covered=34
  AISO               H=2.735  timesteps covered=34


In [12]:
# ── N_TYPES 차원 탐색: AISO 최적 타입 수 찾기 ────────────────────────────────
TEST_TYPES = [4, 6, 8, 9, 12, 14, 17, 20, 24]

def run_aiso_nt(X_anom, n, seed, n_types):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a, N_AG, replace=True)].copy().astype(float)
    W = rng2.dirichlet(np.ones(n_types), N_AG)
    M = rng2.uniform(M_LOW, W_REPEL, (n_types, n_types))
    visit = np.zeros(N_a); w_r = W_REPEL
    for t in range(N_IT):
        if t % 10 == 0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1, N_AG)])
            w_r = 1.0 + 3.0 * np.exp(-div / 0.12)
        C = W @ M @ W.T; np.fill_diagonal(C, 0)
        for i in range(N_AG):
            ci = C[i].copy(); ci[i] = 0
            ta = np.argsort(ci)[-3:]; tr2 = np.argsort(ci)[:3]
            Fv  = sum(ci[j] * (X[j]-X[i]) for j in ta)
            Fv += w_r * sum(ci[j] * (X[j]-X[i]) for j in tr2)
            Fv /= 6.0
            nn = np.argmin(np.linalg.norm(Xn - np.clip(X[i]+ALPHA*Fv, 0, 1), axis=1))
            X[i] = Xn[nn]; visit[nn] += 1.0
            bja = ta[np.argmax(ci[ta])]
            W[i] = (1-BETA)*W[i] + BETA*W[bja]; W[i] /= W[i].sum()
    probs = visit + 1.0; probs /= probs.sum()
    return rng2.choice(N_a, n, replace=True, p=probs)

print('N_TYPES 차원 탐색 (Elliptic Bitcoin)')
print(f'  현재 설정: N_TYPES={N_TYPES}')
print('-'*45)
type_results_nt = {}
for nt in TEST_TYPES:
    idx = run_aiso_nt(X_anom_pca, N_TARGET, SEED, nt)
    Xtr, ytr = build_train(X_norm, X_anom, idx)
    res = evaluate(Xtr, ytr)
    type_results_nt[nt] = res['PR-AUC']
    marker = ' <-- 현재' if nt == N_TYPES else ''
    print(f'  N_TYPES={nt:>2}  PR-AUC={res["PR-AUC"]:.4f}{marker}')

best_nt = max(type_results_nt, key=type_results_nt.get)
print('-'*45)
print(f'  최적 N_TYPES = {best_nt}  (PR-AUC={type_results_nt[best_nt]:.4f})')
print(f'  기본값({N_TYPES}) PR-AUC  = {type_results_nt[N_TYPES]:.4f}')
print(f'  개선 여지        = {type_results_nt[best_nt]-type_results_nt[N_TYPES]:+.4f}')


N_TYPES 차원 탐색 (Elliptic Bitcoin)
  현재 설정: N_TYPES=8
---------------------------------------------
  N_TYPES= 4  PR-AUC=0.7852
  N_TYPES= 6  PR-AUC=0.7802
  N_TYPES= 8  PR-AUC=0.7849 <-- 현재
  N_TYPES= 9  PR-AUC=0.7818
  N_TYPES=12  PR-AUC=0.7811
  N_TYPES=14  PR-AUC=0.7849
  N_TYPES=17  PR-AUC=0.7827


KeyboardInterrupt: 